# MaxKernel · TPU v6e · 五策略完整 ADK 实验

使用 `yezi333/accelerator-agents` 的 `codex/adk-openai-colab` 最新分支。

- 输入沿用 PallasKernels 的 **gqa_decode** 实现和原 YAML 工作负载，作为固定 Pallas baseline。
- 原生 Parallel Search：**5 条显式策略，每条最多 5 轮，最多 2 个 worker 同时运行**。
- 每个 worker 使用原始 AutonomousPipelineAgent：准备、测试生成、规划、实现、编译修复、正确性、调参、Profiling、历史最佳选择。
- 先在 Colab 选择 TPU runtime，按顺序执行 cells。不要在 notebook 主进程初始化 JAX/TPU；TPU 服务负责执行。
- API key 从 Colab Secrets、环境变量或隐藏输入读取。保留原模型与代理地址配置。
- 已有环境可复用；Cell 3 使用快进合并更新 fork，不固定旧 commit。停止旧实验再运行。
- 这是完整原生流程，没有旧实验的 TPU_JOBS、MODEL_CALLS、STRUCTURE_ROUNDS 等简化预算；最多 25 个外层轮次，内部编译/调参另计。每次使用新的实验目录。
- 基线/候选正确性沿用原生 harness，本次更新不修改它的数值检查规则。
- 迭代 LLM 请求的网络等待超时设为 **360 秒（6 分钟）**；SDK 最多额外重试 2 次。它不是整轮优化的时限，也不是包含重试的总墙钟上限。Cell 3.5 的简单连通性测试仍使用 60 秒。


In [ ]:
#@title 1. 完整 ADK：五策略搜索与模型配置
FULL_ITERATIONS = 5 #@param {type:"integer"}
MAX_CONCURRENCY = 2 #@param {type:"integer"}
FULL_RUN_NAME = "gqa-decode-native-five-01" #@param {type:"string"}
STRATEGY_1 = "Loop tiling" #@param {type:"string"}
STRATEGY_2 = "Overlap data movement and compute" #@param {type:"string"}
STRATEGY_3 = "Cache reused data in local memory instead of reloading from main memory" #@param {type:"string"}
STRATEGY_4 = "Loop reordering and restructuring" #@param {type:"string"}
STRATEGY_5 = "Eliminate redundant computation" #@param {type:"string"}
STRATEGIES = [STRATEGY_1, STRATEGY_2, STRATEGY_3, STRATEGY_4, STRATEGY_5]
DOWNLOAD_RESULTS = True #@param {type:"boolean"}

MODEL = "gpt-5.6-sol" #@param {type:"string"}
REQUEST_URL = "https://agent.infiscale-tech.com/v1" #@param {type:"string"}
REASONING_EFFORT = "high" #@param {type:"string"}
# SDK 每次尝试的网络等待超时；完整调用含最多 2 次额外重试。
API_TIMEOUT_SECONDS = 360 #@param {type:"integer"}
MAX_OUTPUT_TOKENS = 8000 #@param {type:"integer"}

import os
import re
from getpass import getpass
from pathlib import Path
from urllib.parse import urlsplit

if not Path("/content").is_dir():
    raise RuntimeError("请在 Google Colab TPU runtime 中运行。")
if not re.fullmatch(r"[A-Za-z0-9_-]+", FULL_RUN_NAME):
    raise ValueError("实验名称只能包含字母、数字、下划线和短横线。")
if any(type(x) is not int or x < 1 for x in [FULL_ITERATIONS, MAX_CONCURRENCY, API_TIMEOUT_SECONDS, MAX_OUTPUT_TOKENS]):
    raise ValueError("轮数、并发、时间和 token 参数必须是正整数。")
STRATEGIES = [s.strip() for s in STRATEGIES]
if len(STRATEGIES) != 5 or len(set(STRATEGIES)) != 5 or not all(STRATEGIES):
    raise ValueError("本实验需要五条不同且非空的策略。")
if not MODEL.strip():
    raise ValueError("MODEL 不能为空。")
base_url = REQUEST_URL.strip().rstrip("/")
if base_url.endswith("/responses"):
    base_url = base_url[:-len("/responses")]
parsed = urlsplit(base_url)
if (parsed.scheme not in {"http", "https"} or not parsed.hostname
    or parsed.username is not None or parsed.password is not None
    or parsed.query or parsed.fragment or parsed.path.endswith("/chat/completions")):
    raise ValueError("REQUEST_URL 必须是 Responses API 基础地址。")

ROOT = Path("/content/maxkernel-colab")
REPO = ROOT / "accelerator-agents"
ADK = REPO / "maxkernel/adk"
PALLAS_REPO = ROOT / "PallasKernels"
ENV_DIR = ROOT / "env"
PYTHON = ENV_DIR / "bin/python"
FULL_WORKDIR = ROOT / "full_adk_results"
RUN_DIR = FULL_WORKDIR / FULL_RUN_NAME

key = os.environ.get("OPENAI_API_KEY", "").strip()
if not key:
    try:
        from google.colab import userdata
        key = (userdata.get("OPENAI_API_KEY") or "").strip()
    except Exception:
        key = ""
if not key:
    key = getpass("请输入 API key（隐藏输入）：").strip()
if not key:
    raise ValueError("API key 不能为空。")
os.environ["OPENAI_API_KEY"] = key
del key
os.environ["MAXKERNEL_MODEL"] = MODEL.strip()
os.environ["OPENAI_BASE_URL"] = base_url
os.environ["MAXKERNEL_REASONING_EFFORT"] = REASONING_EFFORT
os.environ["MAXKERNEL_API_TIMEOUT"] = str(API_TIMEOUT_SECONDS)
os.environ["MAXKERNEL_MAX_OUTPUT_TOKENS"] = str(MAX_OUTPUT_TOKENS)
os.environ["MAXKERNEL_LLM_PROVIDER"] = "openai"
os.environ["MAXKERNEL_ADK_DIR"] = str(ADK)
os.environ["WORKDIR"] = str(FULL_WORKDIR)
os.environ["TPU_VERSION"] = "TPU v6e"
os.environ.pop("RAG_CORPUS", None)
print(f"Parallel Search：{len(STRATEGIES)} 条策略，每条最多 {FULL_ITERATIONS} 轮，并发 {MAX_CONCURRENCY}。")
print("模型：", MODEL)
print("结果目录：", RUN_DIR)
print(f"LLM 网络等待超时：每次尝试 {API_TIMEOUT_SECONDS} 秒；最多额外重试 2 次。")


In [ ]:
#@title 2. 检查配置
required = ["OPENAI_API_KEY", "OPENAI_BASE_URL", "MAXKERNEL_MODEL"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("缺少配置：" + ", ".join(missing))
if RUN_DIR.exists():
    raise RuntimeError("该目录已有实验，请修改 FULL_RUN_NAME。完整流程不做阶段级续跑。")
for i, strategy in enumerate(STRATEGIES, 1):
    print(f"策略 {i}: {strategy}")
print("每条路线均执行：准备 → 测试生成 → 规划/实现/编译修复/正确性/调参/Profile。")
print("编译或正确性失败时，按原 pipeline 跳过本轮后续阶段并进入下一轮。")


In [ ]:
#@title 3. 获取代码并安装环境

import os
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
import zipfile
from pathlib import Path
from google.colab import files


AGENT_BRANCH = "codex/adk-openai-colab"
AGENT_REMOTE = "https://github.com/yezi333/accelerator-agents.git"


def run_live(command, *, env=None):
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
        env=env,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGINT)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=15)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
            raise

        if code:
            raise RuntimeError(f"命令失败，退出码：{code}")


ROOT.mkdir(parents=True, exist_ok=True)

# ---------- 获取最新 fork 分支，保留 Colab 中已有文件 ----------
if not REPO.exists():
    run_live(["git", "clone", "--branch", AGENT_BRANCH, "--single-branch", AGENT_REMOTE, str(REPO)])
else:
    run_live(["git", "-C", str(REPO), "fetch", AGENT_REMOTE, "refs/heads/" + AGENT_BRANCH])
    run_live(["git", "-C", str(REPO), "merge", "--ff-only", "FETCH_HEAD"])
actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
if not (ADK / "experiments/full_adk/search.py").is_file():
    raise RuntimeError("当前分支还没有完整 ADK 多策略入口。")
print("accelerator-agents commit:", actual)

# ---------- 上传和解压 PallasKernels ----------
required_files = [
    "pyproject.toml",
    "pallas_kernels/kernels/gqa_decode.py",
    "pallas_kernels/configs/gqa_decode.yaml",
    "tools/get_tpu_duration_info.py",
    "tools/tpu_duration_profile.py",
]


def valid_source(path):
    return all((path / name).is_file() for name in required_files)


if PALLAS_REPO.exists():
    if not valid_source(PALLAS_REPO):
        raise RuntimeError(
            f"{PALLAS_REPO} 存在但源码不完整，不自动覆盖。"
        )
    print("复用已有 PallasKernels：", PALLAS_REPO)
else:
    print("请上传 PallasKernels ZIP/TAR 文件。")
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError("请只上传一个 ZIP/TAR 文件。")

    archive_path = Path(next(iter(uploaded)))
    del uploaded

    with tempfile.TemporaryDirectory(
        dir=PALLAS_REPO.parent
    ) as directory:
        staging = Path(directory).resolve()

        if zipfile.is_zipfile(archive_path):
            with zipfile.ZipFile(archive_path) as archive:
                for member in archive.infolist():
                    target = (staging / member.filename).resolve()
                    if not target.is_relative_to(staging):
                        raise ValueError("ZIP 包含非法路径。")
                archive.extractall(staging)

        elif tarfile.is_tarfile(archive_path):
            with tarfile.open(archive_path, mode="r:*") as archive:
                archive.extractall(staging, filter="data")

        else:
            raise ValueError("上传文件不是有效 ZIP/TAR。")

        candidates = [
            item.parent
            for item in staging.rglob("pyproject.toml")
            if valid_source(item.parent)
        ]

        if len(candidates) != 1:
            raise RuntimeError(
                "压缩包中未找到唯一且完整的 PallasKernels。"
            )

        shutil.move(str(candidates[0]), str(PALLAS_REPO))

    print("PallasKernels 已解压：", PALLAS_REPO)


# ---------- 安装 Python 3.11 ----------
install_env = os.environ.copy()
install_env.pop("OPENAI_API_KEY", None)
install_env.pop("GITHUB_TOKEN", None)
install_env[
    "SETUPTOOLS_SCM_PRETEND_VERSION_FOR_PALLAS_KERNELS"
] = "0.0.0"

run_live(
    [sys.executable, "-m", "pip", "install", "uv>=0.8,<1"],
    env=install_env,
)

run_live(
    [sys.executable, "-m", "uv", "python", "install", "3.11"],
    env=install_env,
)

if not PYTHON.is_file():
    run_live([
        sys.executable,
        "-m",
        "uv",
        "venv",
        "--python",
        "3.11",
        str(ENV_DIR),
    ], env=install_env)

run_live([
    str(PYTHON),
    "-c",
    "import sys; "
    "assert sys.version_info[:2] == (3, 11), "
    "'需要 Python 3.11'",
], env=install_env)

run_live([
    sys.executable,
    "-m",
    "uv",
    "pip",
    "install",
    "--python",
    str(PYTHON),
    "-r",
    str(ADK / "dependency/openai_requirements.txt"),
    "-e",
    f"{PALLAS_REPO}[tests,tpu]",
], env=install_env)

run_live([
    sys.executable,
    "-m",
    "uv",
    "pip",
    "check",
    "--python",
    str(PYTHON),
], env=install_env)

print("安装完成。")

# 用原 agent_requirements，补上当前 JAX 场景需要的工具依赖。
constraints = ROOT / "full_adk_constraints.txt"
constraints.write_text(
    "jax==0.8.1\njaxlib==0.8.1\ntensorflow==2.20.0\nxprof==2.22.1\n"
    "google-adk==1.39.1\nopenai==2.54.0\n"
)
install_env = os.environ.copy()
install_env.pop("OPENAI_API_KEY", None)
install_env.pop("GITHUB_TOKEN", None)
run_live([
    sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON),
    "-c", str(constraints),
    "-r", str(ADK / "dependency/agent_requirements.txt"),
    "-r", str(ADK / "dependency/openai_requirements.txt"),
    "tensorflow==2.20.0", "xprof==2.22.1", "matplotlib", "pandas",
    "docstring-parser", "tabulate",
], env=install_env)

# 原 MCP 文件工具需要 npx；没有时只装到独立环境。
full_path = str(ENV_DIR / "bin") + os.pathsep + os.environ["PATH"]
if not shutil.which("npx", path=full_path):
    run_live([
        sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON),
        "nodejs-wheel>=20,<25",
    ], env=install_env)

full_env = os.environ.copy()
full_env["PATH"] = full_path
full_env["PYTHONPATH"] = str(ADK) + os.pathsep + str(PALLAS_REPO)
full_env["PYTHONUNBUFFERED"] = "1"
full_env["JAX_PLATFORMS"] = "cpu"  # 仅 agent/协调进程用 CPU。
full_env.pop("JAX_PLATFORM_NAME", None)
full_env.pop("GITHUB_TOKEN", None)
FULL_WORKDIR.mkdir(parents=True, exist_ok=True)

run_live([sys.executable, "-m", "uv", "pip", "check", "--python", str(PYTHON)], env=install_env)
# 在 ADK 启动前预下载原版 MCP 包，避免首次下载超过握手时限。
run_live([
    "npx", "-y", "--package=@modelcontextprotocol/server-filesystem@0.5.1",
    "node", "-e", "console.log('MCP filesystem dependency ready')",
], env=full_env)
# CPU-only 导入检查：提前发现原生 profiling 依赖缺失。
run_live([
    str(PYTHON), "-c",
    "from auto_agent.agent import create_root_agent; "
    "from xprof.convert import raw_to_tool_data; "
    "print('Original ADK and XProf imports ready')",
], env=full_env)
print("完整原生 ADK / Parallel Search 环境准备完成。")

In [ ]:
#@title 3.5. 测试模型 API

import os
import subprocess

test_code = r'''
import os
import sys
from openai import OpenAI, APIConnectionError, APIStatusError

model = os.environ["MAXKERNEL_MODEL"]
api_key = os.environ["OPENAI_API_KEY"]
base_url = os.environ["OPENAI_BASE_URL"]

def safe(value):
    return str(value).replace(api_key, "[REDACTED]")

try:
    with OpenAI(
        api_key=api_key,
        base_url=base_url,
        timeout=60.0,
        max_retries=0,
    ) as client:
        print("测试模型：", model)
        print("请求地址：", base_url + "/responses")

        response = client.responses.create(
            model=model,
            input="Reply with exactly: OK",
            reasoning={"effort": "low"},
            max_output_tokens=1024,
            store=False,
        )

        print("响应状态：", response.status)
        print("返回模型：", response.model)
        print("模型回复：", response.output_text)

        if response.usage:
            print("输入 token：", response.usage.input_tokens)
            print("输出 token：", response.usage.output_tokens)

        if response.status != "completed":
            print("未完成详情：", safe(response.incomplete_details))
            sys.exit(1)

        if not response.output_text.strip():
            print("模型没有返回文本。")
            sys.exit(1)

        print("API 测试通过。")

except APIStatusError as exc:
    print("HTTP 状态码：", exc.status_code)
    print("错误：", safe(exc))
    sys.exit(1)

except APIConnectionError as exc:
    print("连接失败：", safe(exc))
    sys.exit(1)

except Exception as exc:
    print("测试失败：", type(exc).__name__, safe(exc))
    sys.exit(1)
'''

result = subprocess.run(
    [str(PYTHON), "-u", "-c", test_code],
    env=os.environ.copy(),
    text=True,
    capture_output=True,
)

key = os.environ.get("OPENAI_API_KEY", "")
output = (result.stdout or "") + (result.stderr or "")
print(output.replace(key, "[REDACTED]"))

if result.returncode:
    raise RuntimeError("API 测试失败，请不要运行 Cell 4。")

In [ ]:
#@title 4. 启动五策略完整 ADK 搜索
import os
import signal
import subprocess

# 不使用旧 run_experiment 的 SIGKILL 中断方式，给服务清理留时间。
def run_native(command):
    with subprocess.Popen(
        command, env=full_env, cwd=ADK,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except BaseException:
            if process.poll() is None:
                try:
                    os.killpg(process.pid, signal.SIGINT)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=45)
                except subprocess.TimeoutExpired:
                    os.killpg(process.pid, signal.SIGKILL)
                    process.wait()
            raise
        finally:
            # 清理该驱动进程组可能残留的 MCP/npx 子进程。
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        if code:
            raise RuntimeError(f"完整 ADK 运行失败（{code}）；查看上方日志和 {RUN_DIR}/services/*.log。")

if RUN_DIR.exists():
    raise RuntimeError("请使用新的 FULL_RUN_NAME；不要覆盖已有实验。")

run_native([
    str(PYTHON), "-m", "experiments.full_adk.run",
    "--repo", str(PALLAS_REPO),
    "--session-dir", str(RUN_DIR),
    "--iterations", str(FULL_ITERATIONS),
    "--max-concurrency", str(MAX_CONCURRENCY),
    "--strategies", *STRATEGIES,
])
print("搜索已结束；请在下一 cell 查看每条路线的状态和全局最佳结果。")

In [ ]:
#@title 5. 查看各策略、各轮结果及全局最佳
import json

summary_path = RUN_DIR / "search_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print("搜索状态：", summary["status"])
    print("最佳节点：", summary["best_node_id"])
    print("最佳原生 speedup：", summary["best_speedup"])
    print("最佳代码：", summary["optimized_kernel_path"])
    for route in summary["routes"]:
        print(f"\n===== {route['node_id']}: {route['strategy']} =====")
        print("执行状态：", route["execution_status"])
        print("最佳 iteration（从 0 开始）：", route["best_iteration"])
        if route.get("execution_error"):
            print("执行错误：", route["execution_error"])
        for row in route["iterations"]:
            print(f"\n  iteration {row['iteration']} | speedup={row.get('speedup')}")
            print("  编译：", (row.get("compilation_status") or {}).get("success"))
            print("  正确性：", (row.get("test_status") or {}).get("success"))
            if not (row.get("test_status") or {}).get("success"):
                print("  错误：", (row.get("test_status") or {}).get("output", ""))
            print("  Autotuning：", row.get("autotuning_summary", ""))
            print("  Profiling：", row.get("profiling_summary", ""))
        print("  该路线完整会话与文件：", route["session_dir"])
    print("\n原生测试/调参 speedup 不等于独立的 XProf device-only 确认。")
else:
    print("尚无 search_summary.json，检查 services/*.log。")
    graph_path = RUN_DIR / "search_graph.json"
    if graph_path.is_file():
        graph = json.loads(graph_path.read_text())
        for node in graph["nodes"].values():
            if node.get("parent_id") is not None:
                print(node["node_id"], node["strategy_applied"], node["execution_status"], node["evaluation"])


In [ ]:
#@title 6. 下载实验结果

import tarfile
from pathlib import Path

if not DOWNLOAD_RESULTS:
    print("DOWNLOAD_RESULTS=False，跳过下载。")
elif not RUN_DIR.exists():
    print("结果目录不存在。")
else:
    from google.colab import files

    archive_path = ROOT / f"{RUN_DIR.name}.tar.gz"

    def exclude_cache(info):
        if "compilation_cache" in Path(info.name).parts:
            return None
        return info

    with tarfile.open(archive_path, "w:gz") as archive:
        archive.add(
            RUN_DIR,
            arcname=RUN_DIR.name,
            filter=exclude_cache,
        )

    print("结果包：", archive_path)
    files.download(str(archive_path))